# Project SPECTRA — FDIA Pipeline
**Airbus Fly Your Ideas 2025 | IEEE Transactions on Aerospace and Electronic Systems**

### Before running:
1. `Runtime → Change runtime type → T4 GPU`
2. Upload `dftrain.h5`, `dfvalid.h5`, `dfvalid_groundtruth.csv` to your Google Drive
3. Update `DRIVE_DATA_PATH` in Cell 2 to point to the folder containing those files
4. `Runtime → Run all`

In [ ]:
# ============================================================
# CELL 1 — Check GPU
# ============================================================
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# ============================================================
# CELL 2 — Mount Google Drive + set data path
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os

# ---- SET THIS to the folder in your Drive containing the .h5 and .csv files ----
DRIVE_DATA_PATH = '/content/drive/MyDrive/SPECTRA_data'
# ---------------------------------------------------------------------------------

# Verify files are present
required = ['dftrain.h5', 'dfvalid.h5', 'dfvalid_groundtruth.csv']
for f in required:
    path = os.path.join(DRIVE_DATA_PATH, f)
    exists = os.path.exists(path)
    size   = f'{os.path.getsize(path)/1e6:.1f} MB' if exists else 'MISSING'
    print(f'  {f}: {size}')

In [ ]:
# ============================================================
# CELL 3 — Clone repo and install dependencies
# ============================================================
import subprocess

# Clone or pull latest
if not os.path.exists('/content/Project_SPECTRA'):
    !git clone https://github.com/Adhith-Krishna/Project_SPECTRA.git /content/Project_SPECTRA
else:
    !git -C /content/Project_SPECTRA pull

print('\nRepo contents:')
!ls /content/Project_SPECTRA/

# Install any missing packages (h5py, tqdm usually pre-installed on Colab)
!pip install -q h5py tqdm scikit-learn

In [ ]:
# ============================================================
# CELL 4 — Patch pipeline config to use Drive paths
# ============================================================
# We override the DATA_DIR in the pipeline before importing it
# This avoids editing the source file

import sys
sys.path.insert(0, '/content/Project_SPECTRA')

# Monkey-patch the config before the pipeline uses it
import importlib

# Read pipeline source
with open('/content/Project_SPECTRA/fdia_pipeline.py', 'r') as f:
    src = f.read()

# Patch DATA_DIR to point to Drive
src_patched = src.replace(
    'DATA_DIR  = os.path.expanduser(',
    '# DATA_DIR  = os.path.expanduser('
).replace(
    '"~/Airbus_Fly_Your_Ideas_2026/Project_Spectra/data/airbus_heli"',
    '# patched'
)

# Inject correct paths
src_patched = src_patched.replace(
    '# patched\n)',
    f''
)

# Write patched version
patched_path = '/content/fdia_pipeline_colab.py'
with open(patched_path, 'w') as f:
    f.write(src_patched)

# Simpler approach — just set env vars and override at import
os.environ['SPECTRA_DATA_DIR'] = DRIVE_DATA_PATH
print(f'Data dir set to: {DRIVE_DATA_PATH}')

In [ ]:
# ============================================================
# CELL 5 — Run full pipeline (clean import with correct paths)
# ============================================================
# We run the pipeline by exec-ing it with patched DATA_DIR
# This is cleaner than import for a script-style pipeline

with open('/content/Project_SPECTRA/fdia_pipeline.py', 'r') as f:
    pipeline_src = f.read()

# Patch the DATA_DIR line only
pipeline_src = pipeline_src.replace(
    'DATA_DIR  = os.path.expanduser(\n    "~/Airbus_Fly_Your_Ideas_2026/Project_Spectra/data/airbus_heli"\n)',
    f'DATA_DIR  = "{DRIVE_DATA_PATH}"'
)

# Confirm patch worked
for line in pipeline_src.split('\n'):
    if 'DATA_DIR' in line and 'DRIVE' not in line and 'os.path' not in line:
        print(f'Patched: {line.strip()}')
        break

# Execute
exec(compile(pipeline_src, 'fdia_pipeline.py', 'exec'))

In [ ]:
# ============================================================
# CELL 6 — Save outputs to Drive
# ============================================================
import shutil

OUTPUT_DIR = os.path.join(DRIVE_DATA_PATH, 'results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

output_files = [
    'cae_fdia.pt',
    'training_loss.png',
    'roc_curves.png',
    'residual_distributions.png',
    'signal_comparison.png'
]

for fname in output_files:
    if os.path.exists(fname):
        dest = os.path.join(OUTPUT_DIR, fname)
        shutil.copy(fname, dest)
        print(f'Saved: {fname} → Drive/SPECTRA_data/results/')
    else:
        print(f'Not found (may not have been generated): {fname}')

print('\nAll outputs saved to Google Drive.')